Cell 0 — Parameters (Papermill tag: parameters)
This cell is tagged parameters in Jupyter so Papermill can inject overrides here at runtime. That tag is set in the notebook metadata, not in the code itself — in JupyterLab: right-click the cell → Add Tag → type parameters.

In [ ]:
# Cell 0 — tagged: parameters
# Change PART_NAME to switch parts. Papermill can inject it:
#   papermill 01_geometry_openscad.ipynb out.ipynb -p PART_NAME "base_part"
import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")  # ensure src/ is importable inside Docker

PART_NAME = "base_part"
PART_NAME_OVERRIDE = None              # set by Papermill/pipeline_full for sweeps
TIMEOUT_S          = 120

# Derived — do not edit these directly
_pn        = PART_NAME_OVERRIDE if PART_NAME_OVERRIDE else PART_NAME
PARAMS_JSON = f"scad/{_pn}_params.json"
SCAD_FILE   = f"scad/{_pn}.scad"

Cell 1 — Load and validate params

In [2]:
# Cell 1 — Load params and validate schema
from src.geometry.param_schema import PipelineParams

params = PipelineParams.from_json(PARAMS_JSON)

# Apply Papermill override if provided
if PART_NAME_OVERRIDE is not None:
    params.part_name = PART_NAME_OVERRIDE

# Validate — raises AssertionError with a clear message if anything is wrong
params.validate()

print(f"part_name:    {params.part_name}")
print(f"geometry:     {params.geometry}")
print(f"mesh_hints:   {params.mesh_hints}")
print(f"load_hints:   {params.load_hints}")
print(f"export dir:   {params.export.stl_output_dir}")

part_name:    cantilever_arm
geometry:     GeometryParams(length=100.0, width=50.0, height=75.0, wall_hole_diameter=6.0, wall_hole_inset=12.0, load_hole_diameter=8.0, load_hole_offset_z=12.0)
mesh_hints:   MeshHints(target_element_size=20.0, opt_domain_element_size_mm=2.5, refinement_regions=[])
load_hints:   LoadHints(primary_face='right', load_magnitude_n=5000.0)
export dir:   outputs/meshes


Cell 2 — Show OpenSCAD defines
Inspectable checkpoint — you can verify exactly what flags will be passed to OpenSCAD before running it.

In [3]:
# Cell 2 — Inspect defines before running OpenSCAD
defines = params.to_openscad_defines()
print("OpenSCAD -D defines:")
for k, v in defines.items():
    print(f"  {k:<30} = {v}")

OpenSCAD -D defines:
  LENGTH                         = 100.0
  WIDTH                          = 50.0
  HEIGHT                         = 75.0
  WALL_HOLE_DIAMETER             = 6.0
  WALL_HOLE_INSET                = 12.0
  LOAD_HOLE_DIAMETER             = 8.0
  LOAD_HOLE_OFFSET_Z             = 12.0


Cell 3 — Run OpenSCAD

In [4]:
# Cell 3 — Compile geometry
from src.geometry.openscad_runner import run_openscad

result = run_openscad(params, scad_file=SCAD_FILE, timeout_s=TIMEOUT_S)

print(f"Success:    {result.success}")
print(f"Duration:   {result.duration_s}s")
print(f"STL path:   {result.stl_path}")
print(f"STL size:   {result.stl_size_kb} KB")
if result.stderr:
    print(f"\nStderr:\n{result.stderr}")

# Hard fail here — don't let a bad STL silently propagate to Stage 2
result.raise_if_failed()

Success:    True
Duration:   0.478s
STL path:   outputs/meshes/cantilever_arm.stl
STL size:   102.4 KB

Stderr:
Geometries in cache: 10
Geometry cache size in bytes: 43568
CGAL Polyhedrons in cache: 2
CGAL cache size in bytes: 788208
Total rendering time: 0:00:00.246
   Top level object is a 3D object:
   Simple:        yes
   Vertices:      328
   Halfedges:     984
   Edges:         492
   Halffacets:    332
   Facets:        166
   Volumes:         2



Cell 4 — Quick mesh preview
Renders headlessly via PyVista and saves a PNG to outputs/reports/. This gives you a visual sanity check without needing a display.

In [5]:
# Cell 4 — Headless STL preview → outputs/reports/
import os, warnings
import numpy as np
import meshio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path

warnings.filterwarnings(
    "ignore",
    message="overflow encountered in scalar multiply",
    category=RuntimeWarning,
    module=r"meshio\.stl\._stl",
)

m     = meshio.read(str(result.stl_path))
verts = m.points[m.cells_dict["triangle"]]  # (N, 3, 3)
pts   = verts.reshape(-1, 3)

print(f"Vertices:  {len(m.points)}")
print(f"Faces:     {len(m.cells_dict['triangle'])}")
print(f"Bounds:    {[round(v, 2) for v in [pts[:,0].min(), pts[:,0].max(), pts[:,1].min(), pts[:,1].max(), pts[:,2].min(), pts[:,2].max()]]}")

report_dir = Path("outputs/reports")
report_dir.mkdir(parents=True, exist_ok=True)
png_path = report_dir / f"{params.part_name}_geometry.png"

fig = plt.figure(figsize=(7, 5))
ax  = fig.add_subplot(111, projection="3d")
ax.add_collection3d(
    Poly3DCollection(verts, alpha=0.6, facecolor="lightsteelblue", edgecolor="grey", linewidth=0.1)
)
for dim, setter in zip(range(3), [ax.set_xlim, ax.set_ylim, ax.set_zlim]):
    setter(pts[:, dim].min(), pts[:, dim].max())
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.set_title(params.part_name)
plt.tight_layout()
plt.savefig(str(png_path), dpi=150)
plt.close()

print(f"\nPreview saved: {png_path}")

Vertices:  328
Faces:     672
Bounds:    [np.float64(0.0), np.float64(100.0), np.float64(0.0), np.float64(50.0), np.float64(0.0), np.float64(75.0)]

Preview saved: outputs/reports/cantilever_arm_geometry.png


Cell 5 — Export stage output for handoff
Writes a small JSON sidecar next to the STL so Stage 2 (02_mesh_gmsh.ipynb) knows where to find the geometry and what parameters were used, without having to re-read params.json.

In [6]:
# Cell 5 — Write stage handoff sidecar
# 02_mesh_gmsh.ipynb reads this to find the STL and mesh hints
import json
from pathlib import Path
from dataclasses import asdict

handoff = {
    "stage":        "01_geometry",
    "part_name":    params.part_name,
    "stl_path":     str(result.stl_path),
    "stl_size_kb":  result.stl_size_kb,
    "duration_s":   result.duration_s,
    "mesh_hints":   asdict(params.mesh_hints),
    "load_hints":   asdict(params.load_hints),
}

handoff_path = Path(params.export.stl_output_dir) / f"{params.part_name}_stage01.json"
handoff_path.write_text(json.dumps(handoff, indent=2))
print(f"Handoff written: {handoff_path}")
print(json.dumps(handoff, indent=2))

Handoff written: outputs/meshes/cantilever_arm_stage01.json
{
  "stage": "01_geometry",
  "part_name": "cantilever_arm",
  "stl_path": "outputs/meshes/cantilever_arm.stl",
  "stl_size_kb": 102.4,
  "duration_s": 0.478,
  "mesh_hints": {
    "target_element_size": 20.0,
    "opt_domain_element_size_mm": 2.5,
    "refinement_regions": []
  },
  "load_hints": {
    "primary_face": "right",
    "load_magnitude_n": 5000.0
  }
}


How these files connect to the rest of the pipeline
The sidecar JSON from Cell 5 is the contract between Stage 1 and Stage 2. When 02_mesh_gmsh.ipynb runs, its Cell 0 reads outputs/meshes/<part_name>_stage01.json rather than re-parsing params.json — this means even if params change mid-run, Stage 2 always meshes the geometry that was actually compiled, not whatever the current params say.
The mesh_hints.target_element_size value travels through this sidecar into src/meshing/gmsh_pipeline.py in Stage 2, where it controls the global mesh size. The load_hints travel all the way to src/fea/boundary_conditions.py in Stage 3.
Ready to move to Stage 2 — 02_mesh_gmsh.ipynb and src/meshing/?